## Embodiment tutorial: adding a new embodiment (Zarr)

This notebook is a practical tutorial showing how EgoVerse/EgoMimic’s **embodiment abstraction** works (Zarr-only).

You will learn how to:

- define a **new embodiment** (a robot/data “domain”) by implementing a **key map** and a **transform pipeline**
- write a minimal **Zarr episode** for that embodiment
- load that episode with `ZarrEpisode` / `ZarrDataset` so it produces canonical keys like `actions_cartesian`

This notebook intentionally **does not** cover Hydra training / training commands.

### What “embodiment” means in this repo
In this codebase, an *embodiment* is responsible for converting **raw episode store keys** (e.g. `left.cmd_ee_pose`, `images.front_1`) into the **canonical model interface** consumed by downstream code:

- canonical observation keys (e.g. `observations.state.ee_pose`, `observations.images.front_img_1`)
- canonical action keys (e.g. `actions_cartesian`)
- consistent time structure (e.g. action chunk length 100 with interpolation)

Practically, an embodiment is:

- a **key map**: output-key → `{zarr_key, horizon}`
- a **transform list**: post-load transforms applied at dataset read time

We’ll implement a minimal *new bimanual* embodiment called `myrobot_bimanual`.

## High-level data flow

```mermaid
flowchart LR
  zarrEpisode[ZarrEpisode
(folder of arrays + attrs)] --> zarrDataset[ZarrDataset
(key_map reads store keys)]
  zarrDataset --> transforms[Transform_list
(interp/concat/delete/...)]
  transforms --> canonicalBatch[Canonical batch
(e.g. actions_cartesian,
observations.state.ee_pose)]
```

Key places in the code:

- Embodiment registry: `egomimic/rldb/embodiment/embodiment.py`
- Example embodiments:
  - EVA: `egomimic/rldb/embodiment/eva.py`
  - Human/ARIA-style: `egomimic/rldb/embodiment/human.py`
- Transform primitives: `egomimic/rldb/zarr/action_chunk_transforms.py`
- Zarr writer: `egomimic/rldb/zarr/zarr_writer.py`
- Zarr dataset + resolvers: `egomimic/rldb/zarr/zarr_dataset_multi.py`

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

# Notebook-local output directory (you can change this)
OUT_DIR = Path(os.environ.get("EGOVERSE_TUTORIAL_OUT", "./data/tutorial_myrobot_zarr")).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Writing tutorial assets to:", OUT_DIR)

## Step 1 — Write a minimal Zarr episode for `myrobot_bimanual`

We’ll create a tiny synthetic bimanual episode with:

- **store keys** (Zarr arrays):
  - `images.front_1` (JPEG-compressed RGB frames)
  - `left.obs_ee_pose`, `right.obs_ee_pose` (per-frame 7D: xyz + quat(wxyz))
  - `left.obs_gripper`, `right.obs_gripper` (per-frame 1D)
  - `left.cmd_ee_pose`, `right.cmd_ee_pose` (per-frame 7D)
  - `left.gripper`, `right.gripper` (per-frame 1D)

And we’ll set Zarr attrs:

- `embodiment = "myrobot_bimanual"`
- `fps = 30`

Later, the **embodiment key map** will map canonical dataset keys like `observations.images.front_img_1` → store key `images.front_1`.

We use `ZarrWriter` so the output is compatible with `ZarrEpisode` / `ZarrDataset`.

### Reference: expected Zarr store keys (ARIA / EVA)

Use this as a quick checklist when writing / validating episode stores.

#### ARIA (per-frame)
- `[right/left].obs_ee_pose`: `(7,)` = xyz + quat(wxyz)
- `[right/left].obs_keypoints`: `(63,)` = 21 keypoints × 3 (xyz)
- `[right/left].obs_wrist_pose`: `(7,)` = xyz + quat(wxyz)
- `obs_head_pose`: `(7,)` = xyz + quat(wxyz)
- `images.front_1`: `(H, W, 3)` (stored as JPEG)
- `annotations`: `(N,)` list of `{ "text": str, "start_idx": int, "end_idx": int }`

#### EVA (per-frame)
- `[right/left].obs_ee_pose`: `(7,)` = xyz + quat(wxyz)
- `[right/left].cmd_ee_pose`: `(7,)` = xyz + quat(wxyz)
- `[right/left].obs_joints`: `(7,)` = xyz + quat(wxyz)
- `[right/left].cmd_joints`: `(6,)`
- `[right/left].gripper`: `(1,)`
- `images.front_1`: `(H, W, 3)` (stored as JPEG)
- `images.left_wrist`: `(H, W, 3)` (stored as JPEG)
- `images.right_wrist`: `(H, W, 3)` (stored as JPEG)
- `annotations`: `(N,)` list of `{ "text": str, "start_idx": int, "end_idx": int }`

#### Coordinate convention
- right `(+x)`
- down `(+y)`
- forward `(+z)`


In [ ]:
from egomimic.rldb.zarr.zarr_writer import ZarrWriter


def _rand_quat_wxyz(n: int) -> np.ndarray:
    # random unit quaternions in WXYZ
    u1 = np.random.rand(n)
    u2 = np.random.rand(n)
    u3 = np.random.rand(n)
    qx = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    qy = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    qz = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    qw = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return np.stack([qw, qx, qy, qz], axis=-1).astype(np.float32)


def write_synthetic_myrobot_episode(
    *,
    out_root: Path,
    episode_name: str = "myrobot_demo_000000",
    T: int = 90,
    fps: int = 30,
    H: int = 240,
    W: int = 320,
) -> Path:
    ep_path = out_root / f"{episode_name}.zarr"

    # --- images (RGB uint8, T x H x W x 3) ---
    # simple gradient + noise, just to have non-trivial pixels
    base = np.linspace(0, 255, num=W, dtype=np.uint8)[None, None, :, None]
    img = np.repeat(base, H, axis=1)
    img = np.repeat(img, 3, axis=3)
    images_front = np.repeat(img, T, axis=0)
    images_front = (images_front + np.random.randint(0, 10, size=images_front.shape)).clip(
        0, 255
    ).astype(np.uint8)

    # --- poses (xyz + quat(wxyz)) ---
    # left/right observed pose: (T, 7)
    left_xyz = np.cumsum(np.random.randn(T, 3).astype(np.float32) * 0.002, axis=0)
    right_xyz = np.cumsum(np.random.randn(T, 3).astype(np.float32) * 0.002, axis=0) + np.array(
        [0.2, 0.0, 0.0], dtype=np.float32
    )
    left_q = _rand_quat_wxyz(T)
    right_q = _rand_quat_wxyz(T)
    left_obs = np.concatenate([left_xyz, left_q], axis=-1)
    right_obs = np.concatenate([right_xyz, right_q], axis=-1)

    # commanded pose (slightly ahead / smoothed version)
    left_cmd = left_obs.copy()
    right_cmd = right_obs.copy()

    # --- grippers (T,1) ---
    t = np.linspace(0, 2 * np.pi, num=T, dtype=np.float32)
    left_g = (0.5 + 0.5 * np.sin(t))[:, None]
    right_g = (0.5 + 0.5 * np.cos(t))[:, None]

    numeric = {
        "left.obs_ee_pose": left_obs,
        "right.obs_ee_pose": right_obs,
        "left.obs_gripper": left_g,
        "right.obs_gripper": right_g,
        "left.cmd_ee_pose": left_cmd,
        "right.cmd_ee_pose": right_cmd,
        "left.gripper": left_g,
        "right.gripper": right_g,
    }
    images = {
        "images.front_1": images_front,
    }

    # simple language annotation spans (optional)
    annotations = [
        ("reach", 0, max(0, T // 3 - 1)),
        ("grasp", T // 3, max(T // 3, 2 * T // 3 - 1)),
        ("lift", 2 * T // 3, T - 1),
    ]

    ZarrWriter.create_and_write(
        episode_path=ep_path,
        numeric_data=numeric,
        image_data=images,
        embodiment="myrobot_bimanual",
        fps=fps,
        task="synthetic_demo",
        annotations=annotations,
        chunk_timesteps=100,
        enable_sharding=False,
    )
    return ep_path


EP_PATH = write_synthetic_myrobot_episode(out_root=OUT_DIR)
print("Wrote:", EP_PATH)

## Step 2 — Implement the new embodiment

We will add **one enum entry** and **one embodiment class**.

- Add `MYROBOT_BIMANUAL` to `egomimic/rldb/embodiment/embodiment.py`
- Add `egomimic/rldb/embodiment/myrobot.py` implementing:
  - `get_keymap()` (canonical output keys → store keys + horizons)
  - `get_transform_list()` (how raw keys become `actions_cartesian` and `observations.state.ee_pose`)

### What our embodiment will output
For this tutorial we target a **bimanual Cartesian + gripper** canonical interface:

- `observations.images.front_img_1`: (3,H,W) float in [0,1]
- `observations.state.ee_pose`: (14,) = [L xyz ypr g | R xyz ypr g]
- `actions_cartesian`: (100,14) = [L xyz ypr g | R xyz ypr g]

The raw episode stores pose as xyz+quat(wxyz). Our transform list will:

- interpolate commanded poses and grippers from horizon 45 → chunk length 100
- convert quaternions to yaw/pitch/roll (YPR)
- concatenate left+right into the canonical 14D vectors

We’ll implement this using the transform primitives in `egomimic/rldb/zarr/action_chunk_transforms.py`.

In [ ]:
# After you implement the code changes (enum + new embodiment class),
# these imports should work.
from egomimic.rldb.embodiment.embodiment import get_embodiment_id

# NOTE: This will raise until MYROBOT_BIMANUAL is added to the enum.
print("myrobot_bimanual embodiment_id =", get_embodiment_id("myrobot_bimanual"))

In [ ]:
# Once `egomimic/rldb/embodiment/myrobot.py` exists, use it here.
from egomimic.rldb.embodiment.myrobot import MyRobot

key_map = MyRobot.get_keymap()
transform_list = MyRobot.get_transform_list()

print("key_map keys (dataset output keys):")
for k in list(key_map.keys()):
    print("-", k, "->", key_map[k]["zarr_key"], "horizon=", key_map[k].get("horizon"))

print("\n# transforms:")
for t in transform_list:
    print("-", type(t).__name__)

## Step 2b — Walk through the transform list

Once you have a `key_map` and `transform_list`, it’s useful to see *exactly* what each transform does to the batch dict.

In this section we:

- print the transforms **in order**
- trace how keys/shapes change from one transform to the next on a single index

Note: this traces the transform pipeline on **numpy** arrays (pre-torch), matching how `ZarrDataset` applies transforms.

In [ ]:
from __future__ import annotations

from typing import Any

import numpy as np
import simplejpeg


def _shape_dtype(v: Any) -> str:
    if isinstance(v, np.ndarray):
        return f"np{tuple(v.shape)} {v.dtype}"
    if hasattr(v, "shape"):
        try:
            return f"{type(v).__name__}{tuple(v.shape)}"
        except Exception:
            pass
    return type(v).__name__


def _pad_to_horizon(x: np.ndarray, horizon: int) -> np.ndarray:
    if x.shape[0] >= horizon:
        return x
    pad_len = horizon - x.shape[0]
    last = x[-1:]
    pad = np.repeat(last, pad_len, axis=0)
    return np.concatenate([x, pad], axis=0)


def build_pretransform_sample(*, episode_path, key_map: dict, idx: int) -> dict[str, Any]:
    """Minimal reader mirroring ZarrDataset pre-transform behavior.

    Returns dataset-output keys (the key_map keys) mapped to numpy arrays.
    """
    from egomimic.rldb.zarr.zarr_dataset_multi import ZarrEpisode

    ep = ZarrEpisode(episode_path)
    features = ep.metadata.get("features", {}) or {}
    image_keys = {k for k, info in features.items() if info.get("dtype") == "jpeg"}

    out: dict[str, Any] = {}
    for out_key, info in key_map.items():
        zarr_key = info["zarr_key"]
        horizon = info.get("horizon")

        if zarr_key == "annotations":
            # The dataset handles this specially; skip it here to keep the trace simple.
            continue

        if horizon is not None:
            end = min(idx + int(horizon), int(ep.metadata["total_frames"]))
            val = ep.read({zarr_key: (idx, end)})[zarr_key]
            if isinstance(val, np.ndarray):
                val = _pad_to_horizon(val, int(horizon))
        else:
            val = ep.read({zarr_key: (idx, None)})[zarr_key]

        if zarr_key in image_keys:
            decoded = simplejpeg.decode_jpeg(val, colorspace="RGB")
            val = np.transpose(decoded, (2, 0, 1)) / 255.0

        out[out_key] = val

    return out


def trace_transforms(sample: dict[str, Any], transforms: list) -> dict[str, Any]:
    cur = dict(sample)

    print("Transforms (in order):")
    for i, t in enumerate(transforms):
        print(f"  {i:02d}: {type(t).__name__}")

    for i, t in enumerate(transforms):
        before = {k: _shape_dtype(v) for k, v in cur.items()}
        before_keys = set(before.keys())

        cur = t.transform(cur)

        after = {k: _shape_dtype(v) for k, v in cur.items()}
        after_keys = set(after.keys())

        added = sorted(after_keys - before_keys)
        removed = sorted(before_keys - after_keys)
        changed = sorted(k for k in (before_keys & after_keys) if before[k] != after[k])

        print(f"\n[{i:02d}] {type(t).__name__}")
        if added:
            print("  added:", added)
        if removed:
            print("  removed:", removed)
        if changed:
            print("  changed:")
            for k in changed:
                print(f"    - {k}: {before[k]} -> {after[k]}")

    return cur


# Trace on one index (requires `EP_PATH`, `key_map`, `transform_list` defined above)
idx = 0
pre = build_pretransform_sample(episode_path=EP_PATH, key_map=key_map, idx=idx)
print("Pre-transform keys:", sorted(pre.keys()))
post = trace_transforms(pre, transform_list)
print("\nPost-transform keys (subset):")
for k in ["actions_cartesian", "observations.state.ee_pose", "observations.images.front_img_1"]:
    if k in post:
        print("-", k, _shape_dtype(post[k]))

## Step 3 — Load the episode through the new embodiment

Now we instantiate `ZarrDataset` pointing at the episode we wrote.

Important distinction:

- **store keys**: what exists inside the Zarr group (e.g. `images.front_1`, `left.cmd_ee_pose`)
- **output keys**: what the dataset returns to training (e.g. `observations.images.front_img_1`, `actions_cartesian`)

`ZarrDataset` reads **store keys** using the `zarr_key` fields from `key_map`, then applies the `transform_list` to produce canonical **output keys**.

In [ ]:
from egomimic.rldb.zarr.zarr_dataset_multi import ZarrDataset, ZarrEpisode

# Inspect store keys and metadata
zep = ZarrEpisode(EP_PATH)
print("metadata[embodiment] =", zep.metadata.get("embodiment"))

# list a few store keys
store_keys = sorted(list(zep._store.array_keys()))
print("\nstore keys (first 20):")
for k in store_keys[:20]:
    print("-", k)

# Load one sample via ZarrDataset with our key_map + transforms
zds = ZarrDataset(Episode_path=EP_PATH, key_map=key_map, transform_list=transform_list)
item0 = zds[0]

print("\noutput keys (dataset keys):")
for k in sorted(item0.keys()):
    v = item0[k]
    shape = tuple(v.shape) if hasattr(v, "shape") else type(v)
    print(f"- {k:35s} {shape}")

print("\nactions_cartesian:", item0["actions_cartesian"].shape, "(should be (100, 14))")
print("obs ee_pose:", item0["observations.state.ee_pose"].shape, "(should be (14,))")